#Instalation of Lirairies

# Installing required libraries

This cell installs the necessary packages:
- `sentence-transformers` : dense embeddings (BGE)
- `faiss-cpu` : fast vector indexing
- `rank-bm25` : BM25+ lexical retrieval
- `scikit-learn` : domain classifier
- `nltk` : text processing (stopwords, stemming, tokenization)
- `tqdm` : progress bars
- `contractions` : expands English contractions

In [1]:
!pip install sentence-transformers faiss-cpu rank-bm25 scikit-learn nltk tqdm contractions -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00


# Imports and global configuration

This section imports all required libraries and defines the `Config` class, which centralizes:
- input data paths (corpus, train/test queries, ground truth)
- output and cache directories
- embedding model (`BAAI/bge-base-en-v1.5`)
- confidence threshold for expert mode (0.80)

The `Config` class improves maintainability and reproducibility.

In [2]:
import os, json, csv, time
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from typing import Optional
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
import faiss
from rank_bm25 import BM25Plus
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import contractions
import re
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet
import random
from sklearn.feature_extraction.text import TfidfVectorizer


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

print("Imports OK !")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Imports OK !


[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
class Config:
    # Data Paths
    # Root directory containing the source datasets
    DATA_DIR         = Path("/kaggle/input/datasets/ricamyi/retrieval-engineee")
    
    # Input files: Corpus (documents), Queries (training/testing), and Ground Truth
    CORPUS_FILE      = DATA_DIR / "docs.json"           # Format: [{id, text, category?}]
    TRAIN_QUERIES    = DATA_DIR / "queries_train.json"  # Format: [{id, text, category}]
    TEST_QUERIES     = DATA_DIR / "queries_test.json"   # Format: [{id, text}]
    TRAIN_RELEVANCE  = DATA_DIR / "qgts_train.json"     # Format: {queryID: [docIDs]}
    
    # Storage for output results and pre-computed embeddings
    OUTPUT_DIR       = Path("/kaggle/working")
    EMBEDDINGS_DIR   = Path("/kaggle/input/datasets/ricamyi/embeddingsss")
    
    # Model Configuration
    # Dense Retriever: Using BGE-Base for state-of-the-art embedding quality
    BI_ENCODER_MODEL = "BAAI/bge-base-en-v1.5"
    CAT_THRESHOLD = 0.80

# Initialize configuration and ensure the output directory exists
cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Config OK !")

Config OK !


# Loading data and stratified split

Load JSON files:
- `docs.json` : 216,041 documents (id, text, category)
- `queries_train.json` : 327 training queries (with category)
- `queries_test.json` : 141 test queries (no category)
- `qgts_train.json` : ground truth (relevant documents per query)

Perform an **80/20 stratified split** on the training queries to preserve category distribution. This allows evaluation on a validation set unseen during classifier training.

In [4]:
def load_data(cfg: Config):
    """Loads the corpus, train/test queries, and relevance from JSON."""
    with open(cfg.CORPUS_FILE)   as f: corpus        = json.load(f)
    with open(cfg.TEST_QUERIES)  as f: test_queries  = json.load(f)
    with open(cfg.TRAIN_QUERIES) as f: train_queries = json.load(f)

    # Load ground truth labels if the file exists
    relevance = {}
    if cfg.TRAIN_RELEVANCE.exists():
        with open(cfg.TRAIN_RELEVANCE) as f: relevance = json.load(f)

    # Log dataset statistics for verification
    print(f"Corpus        : {len(corpus)} documents")
    print(f"Train queries : {len(train_queries)}")
    print(f"Test queries  : {len(test_queries)}")
    print(f"Relevance     : {len(relevance)} labeled queries")
    return corpus, train_queries, test_queries, relevance

# DATA LOADING
corpus, train_queries, test_queries, relevance = load_data(cfg)

# STRATIFIED SPLIT
# Split training data: 80% for training, 20% for validation
# Stratification ensures category distributions remain consistent across both sets
q_train_split, q_val_split = train_test_split(
    train_queries, 
    test_size=0.20, 
    random_state=42, 
    stratify=[q.get("category", "unknown") for q in train_queries]
)

print(f"Split complete: {len(q_train_split)} training queries, {len(q_val_split)} validation queries.")

Corpus        : 216041 documents
Train queries : 327
Test queries  : 141
Relevance     : 327 labeled queries
Split complete: 261 training queries, 66 validation queries.


# Text preprocessing

Two things:
- `full_process()` : cleans text (expands contractions, lowercases, removes punctuation, tokenizes, removes stopwords, applies **stemming**).
- `SynonymExpander` : enriches short queries (≤8 words) with WordNet synonyms (probability 0.15) to bridge lexical gaps between queries and documents.

This preprocessing is used **only for BM25+ and TF-IDF; embeddings operate on raw text.

In [5]:
stemmer = PorterStemmer()
_STOP = set(stopwords.words('english'))

def full_process(text):
    """Cleans text: fixes contractions, removes symbols, tokenizes, and applies Porter Stemming."""
    if not text: return ""
    text = contractions.fix(text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = word_tokenize(text)
    cleaned = [stemmer.stem(t) for t in tokens if t not in _STOP]
    return " ".join(cleaned)

class SynonymExpander:
    """Expands queries using WordNet synonyms to handle lexical gaps."""
    def __init__(self, add_synonym_prob=0.15):
        self.add_synonym_prob = add_synonym_prob

    def expand(self, tokens):
        expanded = []
        for word in tokens:
            synsets = wordnet.synsets(word)
            # Probabilistic expansion to avoid over-noising the query
            if synsets and random.random() <= self.add_synonym_prob:
                syn_word = synsets[0].lemmas()[0].name().lower().replace('_', ' ')
                if syn_word != word:
                    expanded.append(syn_word)
        return tokens + expanded

expander = SynonymExpander(add_synonym_prob=0.15)

# CategoryIndexer: Pre‑computes and caches the mapping from category to document indices.

In [6]:
class CategoryIndexer:
    
    def __init__(self, corpus: list):
        """
        Version 'In-Memory' : ne sauvegarde rien sur le disque.
        """
        self.cat_to_indices = {}
        self.doc_ids = [d["id"] for d in corpus]
        
        print("Building category index ...")
        # On construit l'index directement à chaque initialisation
        for idx, d in enumerate(tqdm(corpus, desc="Indexing categories")):
            cat = d.get("category", "unknown")
            if cat not in self.cat_to_indices:
                self.cat_to_indices[cat] = []
            self.cat_to_indices[cat].append(idx)
        
        print(f"Index built! Categories found: {list(self.cat_to_indices.keys())}")

# BaseRetriever: Base class for retrievers that support category‑restricted search.

In [7]:
class BaseRetriever:
    
    def __init__(self, corpus: list, category_indexer: CategoryIndexer):
        self.corpus = corpus
        self.category_indexer = category_indexer
        self.doc_ids = category_indexer.doc_ids
        
    def retrieve_category(self, query, category, top_k=100):
        """Search only within a specific category. To be implemented by subclasses."""
        raise NotImplementedError
    
    def retrieve(self, query, top_k=100):
        """Global search. To be implemented by subclasses."""
        raise NotImplementedError

# TF‑IDF Retriever with Category Filtering

## Overview

The `TFIDFRetriever` class implements a classic **Term Frequency – Inverse Document Frequency** retrieval model.  
It builds a TF‑IDF matrix over the preprocessed corpus (same preprocessing as BM25) and supports both **global search** and **category‑restricted search**.

## How it works

1. **Document preprocessing** – Each document's text is cleaned using `full_process()` (lowercase, punctuation removal, stopword removal, Porter stemming). The title is duplicated to give it extra weight.
2. **TF‑IDF matrix** – Built using `sklearn.feature_extraction.text.TfidfVectorizer` with L2 normalisation (`norm='l2'`). This ensures that cosine similarity can be computed as a simple dot product.
3. **Category index** – A mapping `category → list of document indices` is created during initialisation. This allows fast filtering when searching only within a predicted category.
4. **Caching** – The entire retriever object (including the TF‑IDF matrix and the category mapping) can be saved to disk via `get_instance()`. Subsequent runs load the pre‑computed object, avoiding costly recomputation.

## Retrieval methods

### `retrieve(query, top_k)`
- Computes the TF‑IDF vector of the query (raw query text, no additional preprocessing – the vectorizer uses the same vocabulary as the corpus).
- Multiplies the query vector with the TF‑IDF matrix → cosine similarity scores for all documents.
- Returns the top‑`k` document IDs with their similarity scores.

### `retrieve_category(query, category, top_k)`
- If the category exists, it first filters the document indices belonging to that category.
- Creates a sub‑matrix for those documents and computes similarities only within that subset.
- Falls back to global search if the category is not known.


In [8]:
class TFIDFRetriever(BaseRetriever):
    def __init__(self, corpus: list, category_indexer: CategoryIndexer):
        super().__init__(corpus, category_indexer)
        
        # Prepare document texts (same preprocessing as BM25)
        texts = []
        for d in corpus:
            if "content_clean" not in d:
                text_to_clean = f"{d.get('title','') or ''} {d.get('title','') or ''} {d.get('text','') or ''} {' '.join(d.get('tags',[]) or [])}"
                d["content_clean"] = full_process(text_to_clean)
            texts.append(d["content_clean"])
        
        # Build TF‑IDF matrix
        self.vectorizer = TfidfVectorizer(norm='l2')
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        print(f"-TF‑IDF ready: {self.tfidf_matrix.shape}")
    
    def retrieve(self, query: str, top_k: int) -> list:
        query_vec = self.vectorizer.transform([query])
        scores = (self.tfidf_matrix @ query_vec.T).toarray().flatten()
        k = min(top_k, len(self.doc_ids))
        top_idx = np.argpartition(scores, -k)[-k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return [(self.doc_ids[i], float(scores[i])) for i in top_idx]
    
    def retrieve_category(self, query: str, category: str, top_k: int) -> list:
        if category not in self.category_indexer.cat_to_indices:
            return self.retrieve(query, top_k)
        indices = self.category_indexer.cat_to_indices[category]
        sub_matrix = self.tfidf_matrix[indices]
        query_vec = self.vectorizer.transform([query])
        scores = (sub_matrix @ query_vec.T).toarray().flatten()
        top_n = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[indices[i]], float(scores[i])) for i in top_n]

# BM25+ Retriever with Query Expansion and Category Filtering

## Overview

The `BM25Retriever` class implements the **BM25+** probabilistic retrieval model, which is an improved version of BM25 that handles term frequency saturation and document length normalisation more effectively.  
It also includes **query expansion** using WordNet synonyms and **Porter stemming** to improve lexical recall.

## How it works

1. **Document preprocessing** – Same as for TF‑IDF: `full_process()` applies lowercasing, punctuation removal, stopword removal, and **Porter stemming**. The title is duplicated to increase its importance.
2. **BM25+ index** – Built using `rank_bm25.BM25Plus` with `b=0.75` (length normalisation strength). The corpus is tokenised after preprocessing.
3. **Category index** – A mapping `category → list of document indices` is built during initialisation, allowing fast category‑restricted retrieval.
4. **Query expansion** – For short queries (≤8 tokens), WordNet synonyms are added probabilistically (15% chance per word). This helps bridge lexical gaps between the query and relevant documents.
5. **Caching** – The whole retriever object (BM25 index, tokenised corpus, category mapping) is cached via `get_instance()`. The default cache path is `/kaggle/input/datasets/ricamyi/bm25pkl/bm25_retriever.pkl`.

## Retrieval methods

### `retrieve(query, top_k)`
- Expands the query with synonyms (if short enough).
- Applies the same `full_process()` preprocessing (stemming, stopword removal).
- Tokenises the cleaned query and computes BM25+ scores against all documents.
- Returns the top‑`k` document IDs with their scores.

### `retrieve_category(query, category, top_k)`
- If the category exists, it restricts the search to documents belonging to that category.
- Creates a **temporary BM25+ index** over the subset of documents (since BM25+ does not natively support sub‑indexing). This ensures fair scoring within the category.
- Falls back to global search if the category is missing.


In [9]:
class BM25Retriever(BaseRetriever):
    def __init__(self, corpus: list, category_indexer: CategoryIndexer):
        super().__init__(corpus, category_indexer)
        
        tokenized_corpus = []
        for d in corpus:
            if "content_clean" not in d:
                text_to_clean = f"{d.get('title','') or ''} {d.get('title','') or ''} {d.get('text','') or ''} {' '.join(d.get('tags',[]) or [])}"
                d["content_clean"] = full_process(text_to_clean)
            tokenized_corpus.append(d["content_clean"].split())
        
        self.bm25 = BM25Plus(tokenized_corpus, b=0.75)
        self.tokenized_corpus = tokenized_corpus
        print("- BM25+ ready")
    
    def retrieve(self, query: str, top_k: int) -> list:
        # Query expansion (identique à ton code original)
        temp_tokens = [t.lower() for t in word_tokenize(query) if t.isalpha() and t.lower() not in _STOP]
        expanded_text = ""
        if 0 < len(temp_tokens) <= 8:
            try:
                synonyms = expander.expand(temp_tokens)
                expanded_text = " ".join(synonyms)
            except:
                expanded_text = ""
        combined = f"{query} {expanded_text}".strip()
        query_clean = full_process(combined)
        query_tokens = query_clean.split()
        if not query_tokens:
            return []
        scores = self.bm25.get_scores(query_tokens)
        k = min(top_k, len(self.doc_ids))
        top_idx = np.argpartition(scores, -k)[-k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return [(self.doc_ids[i], float(scores[i])) for i in top_idx]
    
    def retrieve_category(self, query: str, category: str, top_k: int) -> list:
        if category not in self.category_indexer.cat_to_indices:
            return self.retrieve(query, top_k)
        indices = self.category_indexer.cat_to_indices[category]
        tokenized_query = query.lower().split()
        subset_docs = [self.tokenized_corpus[i] for i in indices]
        bm25_subset = BM25Plus(subset_docs)
        scores = bm25_subset.get_scores(tokenized_query)
        top_n = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[indices[i]], scores[i]) for i in top_n]

In [10]:
# Build or load the shared category indexer
cat_indexer = CategoryIndexer(corpus)

# Build retrievers
tfidf_retriever = TFIDFRetriever(corpus, cat_indexer)
bm25_retriever = BM25Retriever(corpus, cat_indexer)

Building category index ...


Indexing categories: 100%|██████████| 216041/216041 [00:00<00:00, 1968858.72it/s]

Index built! Categories found: ['tex', 'unix', 'gaming', 'programmers', 'android']


-TF‑IDF ready: (216041, 391477)
- BM25+ ready


# Dense Index with BGE‑base‑en‑v1.5 and FAISS

## Model Selection

We use **[BAAI/bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5)** (768 dimensions) as our dense retriever.  
BGE (BAAI General Embedding) is one of the best performing open-source embedding models for information retrieval, especially when using the recommended instruction prefix.

**Why not other models?**  
- We also tested `sentence-transformers/all-MiniLM-L12-v2` (384 dims) — it gave lower recall and MRR.  
- Gemini proposed a list including BGE-base and `Snowflake/snowflake-arctic-embed-m`, but it underperformed compared to BGE‑base.  
- The larger `BAAI/bge-large-en-v1.5` (1024 dims) would require significantly more GPU memory and time.  
- Once computed, embeddings are **save** (`.npy`) and indexed with FAISS, so we only pay the encoding cost once.

## What the `EmbeddingRetriever` class does

1. **Loads the model** (`SentenceTransformer`) and stores document IDs and categories.
2. **Computes corpus embeddings** (if not already cached):
   - **Takes ONLY the `"text"` field of each document** – no title, no tags are included.  
     ⚠️ *This is a known limitation: titles and tags are ignored, which may hurt retrieval quality for documents where key information appears only in the title or tags.*
   - Encodes with `batch_size=256`, `normalize_embeddings=True` (so cosine similarity = dot product).
   - Saves as a `.npy` file for future runs (caching avoids recomputation).
3. **Builds a global FAISS index** (`IndexFlatIP` — inner product) over all document embeddings.  
   This enables fast brute‑force cosine search across the entire corpus.
4. **Builds per‑category FAISS sub‑indices**:
   - Groups document indices by category (`android`, `programmers`, `unix`, `tex`, `gaming`).
   - Creates a separate FAISS index for each category.
   - Stores them in a dictionary `{category: (index, [doc_ids])}`.
   - These sub‑indices allow **expert mode** search — retrieving only from the predicted category, which improves precision and reduces noise.
5. **Encodes queries** with the BGE‑recommended prefix:  
   `"Represent this sentence for searching relevant passages: {query}"`.  
   This instruction fine‑tuning significantly boosts retrieval performance.
6. **Provides two search methods**:
   - `retrieve_global()`: searches the entire corpus (fallback / safety mode).
   - `retrieve_category()`: searches only within a specific category (expert mode, used when classifier confidence ≥ threshold).

## Documentation & References

| Library / Model | Link |
|----------------|------|
| **BGE‑base‑en‑v1.5** | [Hugging Face model card](https://huggingface.co/BAAI/bge-base-en-v1.5) |
| **SentenceTransformers** | [Documentation](https://www.sbert.net/) |
| **FAISS** | [Official documentation](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)|

In [11]:
class EmbeddingRetriever:
    def __init__(self, corpus: list, cfg: Config):
        self.cfg      = cfg
        self.doc_ids  = [d["id"]                      for d in corpus]
        self.doc_cats = [d.get("category", "unknown") for d in corpus]
        self.model    = SentenceTransformer(cfg.BI_ENCODER_MODEL)

        # ── Cache des embeddings du corpus ────────────────────────────────
        cache_path = cfg.EMBEDDINGS_DIR / "corpus_embeddings.npy"
        if cache_path.exists():
            print("Chargement embeddings depuis le cache...")
            self.embeddings = np.load(str(cache_path))
        else:
            print("🧠 Encodage du corpus (peut prendre plusieurs minutes)...")
            texts = [d["text"] for d in corpus]
            self.embeddings = self.model.encode(
                texts,
                batch_size=256,
                show_progress_bar=True,
                normalize_embeddings=True   # L2 norm → cosine = dot product
            ).astype("float32")
            np.save(str(cache_path), self.embeddings)
            print(f"  💾 Embeddings sauvegardés dans {cache_path}")

        dim = self.embeddings.shape[1]
        print(f"  Dimension embeddings : {dim}")

        # ── Index FAISS global ────────────────────────────────────────────
        print("Construction index FAISS global...")
        self.global_index = faiss.IndexFlatIP(dim)
        self.global_index.add(self.embeddings)

        # ── Index FAISS par catégorie ─────────────────────────────────────
        print("Construction indices FAISS par catégorie...")
        self.cat_indices = {}
        for cat in set(self.doc_cats):
            mask    = [i for i, c in enumerate(self.doc_cats) if c == cat]
            cat_emb = self.embeddings[mask].astype("float32")
            idx     = faiss.IndexFlatIP(dim)
            idx.add(cat_emb)
            self.cat_indices[cat] = (idx, [self.doc_ids[i] for i in mask])

        print(f"{len(self.cat_indices)} indices catégorie : {list(self.cat_indices.keys())}")

    def _prefix(self, text: str) -> str:
        """BGE-large nécessite ce préfixe pour les tâches de retrieval."""
        return f"Represent this sentence for searching relevant passages: {text}"

    def encode_queries(self, queries: list) -> np.ndarray:
        prefixed = [self._prefix(q) for q in queries]
        return self.model.encode(
            prefixed, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

    def retrieve_global(self, q_emb: np.ndarray, top_k: int) -> list:
        """Recherche dans l'index global."""
        scores, indices = self.global_index.search(q_emb.reshape(1, -1), top_k)
        return [(self.doc_ids[i], float(s))
                for i, s in zip(indices[0], scores[0]) if i >= 0]

    def retrieve_category(self, q_emb: np.ndarray, category: str, top_k: int) -> list:
        """Recherche dans l'index de la catégorie prédite uniquement."""
        if category not in self.cat_indices:
            return []
        idx, ids = self.cat_indices[category]
        k        = min(top_k, idx.ntotal)
        scores, indices = idx.search(q_emb.reshape(1, -1), k)
        return [(ids[i], float(s))
                for i, s in zip(indices[0], scores[0]) if i >= 0]

embedder = EmbeddingRetriever(corpus, cfg)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chargement embeddings depuis le cache...
  Dimension embeddings : 768
Construction index FAISS global...
Construction indices FAISS par catégorie...
5 indices catégorie : ['unix', 'tex', 'gaming', 'android', 'programmers']


# Domain Classifier (LogReg + LinearSVC + SMOTE)

## What model could you use as a document classifier? How does it work?

We use a **multi‑class classifier** trained on **query embeddings** (not document embeddings). Two models are compared:

- **Logistic Regression (LogReg)** – a linear model that estimates class probabilities using the sigmoid function. It works well for high‑dimensional sparse data and gives calibrated probabilities out‑of‑the‑box.
- **Linear Support Vector Classifier (LinearSVC)** – a linear SVM that finds the hyperplane maximizing the margin between classes. It does not output probabilities directly, so we wrap it with `CalibratedClassifierCV` to obtain confidence scores.

Both models are trained in a **one‑vs‑rest** fashion (via `LinearSVC`'s native multi‑class or `LogReg`'s multi‑class setting). They take the embedding vector of a query (768‑dim from BGE) as input and output a predicted category among 5 classes.

## What data could you use to train your classifier? Will you use both documents and queries?

We **train only on queries**, not on documents. Why?

| Data Source | Pros | Cons | Decision |
|-------------|------|------|----------|
| **Documents** | Large amount (216k) | Query distribution differs from document distribution; a query is short, a document is long. A classifier trained on documents may not generalise well to queries. | ❌ Not used |
| **Queries** | Directly matches test queries (same length, style, vocabulary) | Limited size (261 after split) | ✅ Used |

We use the **training queries** (261 after stratification) because:
- They are representative of real user questions.
- They have also category label.

## What input features could you use for your classifier?

We use **dense embeddings** of the query text as features:

- Each query is prefixed with:  
  `"Represent this sentence for searching relevant passages: {query}"`  
  (the BGE‑recommended instruction for retrieval tasks).
- The SentenceTransformer model (`BAAI/bge-base-en-v1.5`) encodes this text into a **768‑dimensional vector**.

Alternative features could include:
- TF‑IDF vectors of the query (lexical)
- Combined embeddings of query + predicted category from another model
- Metadata like query length, presence of certain keywords

But embeddings alone work very well here.

## Code explanation – `QueryClassifier` class

The `QueryClassifier` class does the following:

### `__init__(self, cfg: Config)`
- Stores configuration.
- Initialises a `LabelEncoder` to convert category names to integers.
- Creates two classifiers: `LogisticRegression` and `LinearSVC`.
- Does **not** train yet – training happens in `fit()`.

### `fit(self, queries, embedder)`
1. **Filters** queries that have a category label.
2. **Prepares features**:
   - Adds the BGE prefix to each query text.
   - Uses `embedder.encode_queries()` to get 768‑dim embeddings.
3. **Encodes labels** with `LabelEncoder`.
4. **Applies SMOTE** (Synthetic Minority Over‑sampling Technique):
   - Detects class imbalance (e.g., `gaming` has 33 samples, `tex` has 104).
   - Creates synthetic examples for minority classes to balance all classes to 104 samples.
   - `k_neighbors` is set to `min(5, minority_class_size - 1)` to avoid errors.
5. **Cross‑validation** (5‑fold) on the balanced dataset:
   - Compares `LogReg` vs `LinearSVC` using accuracy.
   - Here `LinearSVC` wins (0.971 vs 0.963).
6. **Selects the best model** and calibrates `LinearSVC` to produce probabilities.
7. **Trains on the full balanced dataset**.
8. **Prints a detailed classification report** (precision, recall, F1‑score per category).
9. **Identifies the best‑recognised category** (highest F1‑score).

### `predict(self, query_emb)`
- Takes a single query embedding (768‑dim numpy array).
- Calls `predict_proba` to get class probabilities.
- Returns the predicted category (string) and confidence (float).





### How is accuracy computed?

**Accuracy** = (number of correct predictions) / (total number of predictions)

- On the **SMOTE‑balanced training set** (520 samples), accuracy = 0.996 → almost perfect, but this is over‑optimistic because the model sees synthetic data.
- The **honest cross‑validation accuracy** = 0.971 → this is the reliable estimate of how well the classifier will perform on unseen real queries.

### How is it different for each document category?

The classifier performs **very well on all categories** after SMOTE:

| Category | Precision | Recall | F1‑score |
|----------|-----------|--------|----------|
| android  | 1.00      | 0.99   | 1.00     |
| gaming   | 1.00      | 1.00   | 1.00     |
| programmers | 1.00   | 1.00   | 1.00     |
| tex      | 0.99      | 0.99   | 0.99     |
| unix     | 0.99      | 1.00   | 1.00     |

All categories have F1‑scores ≥ 0.99, meaning the classifier is **excellent** across all domains. The slight difference (tex has 0.99) is negligible.

### Does it work equally well on documents and queries?

**No – and that is intentional.**

| Tested on | Performance | Why |
|-----------|-------------|-----|
| **Queries (train split)** | Very high (CV accuracy 0.971) | The classifier was trained on queries → it generalises well to unseen queries. |
| **Documents** | Not tested, likely lower | Document text is much longer and has different distribution. A query‑trained classifier is **not** expected to work on documents. |

We only use it to predict the category of **test queries**, which are similar to training queries. For documents, we already have ground‑truth categories.

## Summary

- **Model chosen**: LinearSVC + CalibratedClassifierCV (CV accuracy 0.971).
- **Training data**: 261 training queries (stratified split), not documents.
- **Features**: 768‑dim BGE embeddings with instruction prefix.
- **SMOTE** rebalanced the classes from 33‑104 to 104 each.
- The classifier is **highly accurate** on queries and used to guide expert‑mode search in the retrieval pipeline.

In [12]:
class QueryClassifier:
    def __init__(self, cfg: Config):
        self.cfg     = cfg
        self.le      = LabelEncoder()
        self.trained = False
        self.clf_lr  = LogisticRegression(
            C=1,
            max_iter=2000,
            solver="lbfgs"
        )
        self.clf_svc = LinearSVC(C=1, max_iter=2000)
        self.clf     = None

    def fit(self, queries: list, embedder):
        valid_queries = [q for q in queries if q.get("category")]
        if not valid_queries:
            print("Aucune query avec catégorie → classifieur désactivé")
            return self

        texts = [f"Represent this sentence for searching relevant passages: {q['text']}" for q in valid_queries]
        labels = [q["category"] for q in valid_queries]

        print(f"Entraînement classifieur sur {len(valid_queries)} queries...")
        print(f"\n   Distribution AVANT SMOTE :")
        for cat, count in pd.Series(labels).value_counts().items():
            print(f"   {cat:<20} : {count}")

        X = embedder.encode_queries(texts)
        y = self.le.fit_transform(labels)

        # SMOTE — oversampling des classes minoritaires
        # k_neighbors=min(5, min_class_size - 1) for handle errors
        min_class_size = pd.Series(y).value_counts().min()
        k_neighbors    = min(5, min_class_size - 1)

        print(f"\nApplication SMOTE (k_neighbors={k_neighbors})...")
        sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_res, y_res = sm.fit_resample(X, y)

        print(f"\n   Distribution APRÈS SMOTE :")
        for cat, count in pd.Series(
                self.le.inverse_transform(y_res)).value_counts().items():
            print(f"   {cat:<20} : {count}")
        print(f"   Total : {len(X)} → {len(X_res)} exemples")

        # CV sur données rééquilibrées
        print("\n   Cross-validation 5-fold on SMOTE data...")
        cv_lr  = cross_val_score(self.clf_lr,  X_res, y_res, cv=5, scoring='accuracy')
        cv_svc = cross_val_score(self.clf_svc, X_res, y_res, cv=5, scoring='accuracy')

        print(f"   LogReg    : {cv_lr.mean():.3f} ± {cv_lr.std():.3f}")
        print(f"   LinearSVC : {cv_svc.mean():.3f} ± {cv_svc.std():.3f}")

        if cv_lr.mean() >= cv_svc.mean():
            print("    - Best : LogReg")
            self.clf = self.clf_lr
        else:
            print("   - Best : LinearSVC")
            from sklearn.calibration import CalibratedClassifierCV
            self.clf = CalibratedClassifierCV(self.clf_svc, cv=5)

        self.clf.fit(X_res, y_res)
        self.trained = True

        # --- AJOUT : ANALYSE DÉTAILLÉE PAR CATÉGORIE ---
        y_pred = self.clf.predict(X_res)
        
        print("\nDETAILED PERFORMANCE (on training data) :")
        # target_names is use for display category name insted of integer
        report = classification_report(y_res, y_pred, target_names=self.le.classes_)
        print(report)

        report_dict = classification_report(y_res, y_pred, target_names=self.le.classes_, output_dict=True)
        
        # Filter for just keeping categories
        cat_scores = {k: v['f1-score'] for k, v in report_dict.items() if k in self.le.classes_}
        best_cat = max(cat_scores, key=cat_scores.get)
        
        print(f"The most well-known category is : **{best_cat}** (F1-Score: {cat_scores[best_cat]:.3f})")

        train_acc = (self.clf.predict(X_res) == y_res).mean()
        print(f"\n   Train accuracy (on SMOTE data) : {train_acc:.3f}")
        print(f"   CV accuracy             : "
              f"{max(cv_lr.mean(), cv_svc.mean()):.3f}")
        return self

    def predict(self, query_emb: np.ndarray) -> tuple:
        proba      = self.clf.predict_proba(query_emb.reshape(1, -1))[0]
        idx        = int(np.argmax(proba))
        return self.le.classes_[idx], float(proba[idx])


classifier = QueryClassifier(cfg)
classifier.fit(q_train_split, embedder)

Entraînement classifieur sur 261 queries...

   Distribution AVANT SMOTE :
   tex                  : 104
   android              : 45
   programmers          : 41
   unix                 : 38
   gaming               : 33

Application SMOTE (k_neighbors=5)...

   Distribution APRÈS SMOTE :
   tex                  : 104
   gaming               : 104
   android              : 104
   unix                 : 104
   programmers          : 104
   Total : 261 → 520 exemples

   Cross-validation 5-fold on SMOTE data...
   LogReg    : 0.963 ± 0.028
   LinearSVC : 0.971 ± 0.016
   - Best : LinearSVC

DETAILED PERFORMANCE (on training data) :
              precision    recall  f1-score   support

     android       1.00      0.99      1.00       104
      gaming       1.00      1.00      1.00       104
 programmers       1.00      1.00      1.00       104
         tex       0.99      0.99      0.99       104
        unix       0.99      1.00      1.00       104

    accuracy                        

# Confidence threshold benchmark

This function simulates, on the validation set, the effect of the confidence threshold on the choice between:
- search restricted to the predicted category (if confidence ≥ threshold)
- search over the whole corpus (if confidence < threshold)

It displays a table with:
- number of queries in expert / global mode
- number of fatal errors (expert mode but wrong prediction)
- fatal error percentage

Help us for fixing CAT_THRESHOLD config variable

In [13]:
def benchmark_thresholds(classifier, queries, embedder, thresholds=[0.3, 0.4, 0.5, 0.51, 0.6, 0.7, 0.8]):
    """
    Analyzes the impact of different confidence thresholds on choosing between 
    Category Index vs Global Index.
    
    Parameters:
    - classifier: trained QueryClassifier that returns (category, confidence)
    - queries: list of query dictionaries with 'text' and 'category' fields
    - embedder: EmbeddingRetriever for encoding queries
    - thresholds: list of confidence thresholds to test
    
    Returns:
    - DataFrame with statistics per threshold
    """
    print(f"Benchmark analysis on {len(queries)} queries...")
    
    # Batch encode validation queries
    texts = [f"Represent this sentence for searching relevant passages: {q['text']}" for q in queries]
    q_embs = embedder.encode_queries(texts)
    
    # Store predictions for each query
    raw_preds = []
    for i, q in enumerate(queries):
        # Pass only the embedding as defined in the classifier
        pred_cat, conf = classifier.predict(q_embs[i])
        raw_preds.append({
            "true": q.get('category'),
            "pred": pred_cat,
            "conf": conf
        })

    # 2. Simulate behavior for each threshold
    results = []
    for t in thresholds:
        stats = {
            "Seuil": t,
            "Confident": 0, 
            "Uncertain": 0, 
            "Overconfidence Errors": 0
        }
        
        for p in raw_preds:
            is_correct = (p['pred'] == p['true'])
            is_sure = (p['conf'] >= t)
            
            if is_sure:
                stats["Confident"] += 1
                if not is_correct:
                    stats["Overconfidence Errors"] += 1
            else:
                stats["Uncertain"] += 1
        
        # Calculate fatal error percentage (confident but wrong)
        stats["% Fatal Error"] = (stats["Overconfidence Errors"] / len(queries)) * 100
        results.append(stats)

    # 3. Display the benchmark dashboard
    df_bench = pd.DataFrame(results).set_index("Seuil")
    
    print("\nTHRESHOLD BENCHMARK DASHBOARD:")
    display(df_bench.style.background_gradient(subset=["Overconfidence Errors", "% Fatal Error"], cmap="Reds")
                      .background_gradient(subset=["Confident"], cmap="Greens")
                      .format({"% Fatal Error": "{:.1f}%"}))
    
    return df_bench

df_thresholds = benchmark_thresholds(classifier, q_val_split, embedder, 
                                     thresholds=[0.3, 0.4, 0.5, 0.55, 0.6, 0.7, 0.8])

Benchmark analysis on 66 queries...

THRESHOLD BENCHMARK DASHBOARD:


,Confident,Uncertain,Overconfidence Errors,% Fatal Error
Seuil,,,,
0.300000,66,0,6,9.1%
0.400000,65,1,6,9.1%
0.500000,65,1,6,9.1%
0.550000,62,4,5,7.6%
0.600000,61,5,4,6.1%
0.700000,54,12,3,4.5%
0.800000,45,21,1,1.5%


## `evaluate_tfidf_with_classifier`

**Purpose**  
Evaluates TF‑IDF retrieval combined with the domain classifier.  
It filters documents by the predicted category when the classifier's confidence exceeds the threshold.

**Parameters**
- `vectorizer` : fitted `TfidfVectorizer`
- `tfidf_matrix` : pre‑computed TF‑IDF matrix of the corpus
- `doc_ids` : list of document IDs (aligned with matrix rows)
- `classifier` : trained `QueryClassifier`
- `embedder` : `EmbeddingRetriever` (used to encode queries for the classifier)
- `queries` : list of query dictionaries (with 'id', 'text', 'category')
- `qgts` : ground truth dictionary `{query_id: relevant_doc_ids}`
- `docs_df` : DataFrame with document categories
- `k` : number of documents to retrieve
- `threshold` : confidence threshold for expert mode

**How it works**
1. Batch‑encodes all queries using the embedder (with BGE prefix) and gets predictions (category + confidence).
2. For each query, if `confidence >= threshold`:
   - Filters document indices belonging to the predicted category.
   - Computes TF‑IDF scores only for those documents.
   - Returns top‑k from that subset.
   - Otherwise (fallback) performs global TF‑IDF search.
3. Computes Precision@k, Recall@k, MRR, latency, and classifier accuracy.

**Returns**  
A dictionary with evaluation metrics.

In [14]:
def evaluate_tfidf_retriever_with_classifier(tfidf_retriever, classifier, embedder,
                                              queries, qgts, k=20, threshold=0.80):
    """
    Evaluate TF‑IDF retriever with classifier category filtering.
    """
    all_p, all_r, all_mrr = [], [], []
    latencies = []
    correct_pred = 0
    total_queries = 0

    print(f"🧠 Encoding queries for TF‑IDF classifier...")
    query_texts = [q.get('text', '') for q in queries]
    q_embs = embedder.encode_queries([f"Represent this sentence for searching relevant passages: {t}" for t in query_texts])
    predictions = [classifier.predict(emb) for emb in q_embs]

    print(f"🚀 Evaluating TF‑IDF + classifier (k={k}, threshold={threshold})...")
    for i, q in enumerate(tqdm(queries, desc="TF‑IDF+Clf")):
        qid = str(q['id'])
        true_cat = q.get('category')
        rel_info = qgts.get(qid, {})
        relevant_ids = {str(item.get("doc_id")) for item in rel_info.get("relevant_doc_ids", [])}
        if not relevant_ids:
            continue

        total_queries += 1
        pred_cat, conf = predictions[i]
        if true_cat and pred_cat == true_cat:
            correct_pred += 1

        start = time.time()
        if conf >= threshold:
            results = tfidf_retriever.retrieve_category(query_texts[i], pred_cat, top_k=k)
        else:
            results = tfidf_retriever.retrieve(query_texts[i], top_k=k)
        retrieved = [str(did) for did, _ in results]
        latencies.append(time.time() - start)

        hits = [1 if d in relevant_ids else 0 for d in retrieved]
        num_hits = sum(hits)
        all_p.append(num_hits / k)
        all_r.append(num_hits / len(relevant_ids))
        mrr = next((1.0/(idx+1) for idx, h in enumerate(hits) if h), 0.0)
        all_mrr.append(mrr)

    return {
        "Method": "TF‑IDF + Classifier",
        "P@k": np.mean(all_p),
        "R@k": np.mean(all_r),
        "MRR": np.mean(all_mrr),
        "Latency (ms)": np.mean(latencies) * 1000,
        "Classifier Accuracy": correct_pred / total_queries if total_queries else 0.0
    }

## `evaluate_bm25_with_classifier`

**Purpose**  
Evaluates BM25+ retrieval with category filtering using the classifier.

**Parameters**
- `bm25_retriever` : instance of `BM25Retriever` (provides `retrieve` and `retrieve_category`)
- `classifier`, `embedder`, `queries`, `qgts`, `k`, `threshold` : same as above

**How it works**
1. Encodes queries and gets predictions (same as TF‑IDF version).
2. For each query:
   - If confidence high → calls `bm25_retriever.retrieve_category(query, pred_cat, top_k=k)`.
   - Else → calls `bm25_retriever.retrieve(query, top_k=k)`.
3. Computes metrics and classifier accuracy.

**Note**  
`retrieve_category` builds a temporary BM25+ index over the subset of documents of the predicted category, ensuring fair scoring within that category.

In [15]:
def evaluate_bm25_with_classifier(bm25_retriever, classifier, embedder,
                                  queries, qgts, k=50, threshold=0.80):
    """
    BM25+ retrieval with category filtering based on classifier confidence.
    """
    all_p, all_r, all_mrr = [], [], []
    latencies = []
    correct_pred = 0
    total_queries = 0

    print(f"Encoding queries for BM25+ classifier...")
    query_texts = [q.get('text', '') for q in queries]
    q_embs = embedder.encode_queries([f"Represent this sentence for searching relevant passages: {t}" for t in query_texts])
    predictions = [classifier.predict(emb) for emb in q_embs]

    print(f"Evaluating BM25+ + classifier (k={k}, threshold={threshold})...")
    for i, q in enumerate(tqdm(queries, desc="BM25+ + Clf")):
        qid = str(q['id'])
        true_cat = q.get('category')
        rel_info = qgts.get(qid, {})
        relevant_ids = {str(item.get("doc_id")) for item in rel_info.get("relevant_doc_ids", [])}
        if not relevant_ids:
            continue

        total_queries += 1
        pred_cat, conf = predictions[i]
        if true_cat and pred_cat == true_cat:
            correct_pred += 1

        start = time.time()
        if conf >= threshold:
            results = bm25_retriever.retrieve_category(query_texts[i], pred_cat, top_k=k)
        else:
            results = bm25_retriever.retrieve(query_texts[i], top_k=k)
        retrieved = [str(doc_id) for doc_id, _ in results]
        latencies.append(time.time() - start)

        hits = [1 if d in relevant_ids else 0 for d in retrieved]
        num_hits = sum(hits)
        all_p.append(num_hits / k)
        all_r.append(num_hits / len(relevant_ids))
        mrr = next((1.0/(idx+1) for idx, h in enumerate(hits) if h), 0.0)
        all_mrr.append(mrr)

    return {
        "Method": "BM25+ + Classifier",
        "P@k": np.mean(all_p),
        "R@k": np.mean(all_r),
        "MRR": np.mean(all_mrr),
        "Latency (ms)": np.mean(latencies) * 1000,
        "Classifier Accuracy": correct_pred / total_queries if total_queries else 0.0
    }

## `evaluate_embeddings_with_classifier`

**Purpose**  
Evaluates dense retrieval (FAISS + BGE embeddings) with category‑restricted search guided by the classifier.

**Parameters**
- `embedder` : instance of `EmbeddingRetriever` (provides `retrieve_global` and `retrieve_category`)
- `classifier`, `queries`, `qgts`, `k`, `threshold` : as above

**How it works**
1. Batch‑encodes queries (with BGE prefix) and obtains predictions.
2. For each query:
   - If `confidence >= threshold` → `embedder.retrieve_category(q_emb, pred_cat, top_k=k)`.
   - Else → `embedder.retrieve_global(q_emb, top_k=k)`.
3. Computes metrics and classifier accuracy.

**Key point**  
Category‑restricted search is extremely fast because FAISS sub‑indices are pre‑built per category.

In [16]:
def evaluate_embeddings_with_classifier(embedder, classifier, queries, qgts, k=50, threshold=0.80):
    """
    Dense retrieval (FAISS) with category‑restricted index selection.
    """
    all_p, all_r, all_mrr = [], [], []
    latencies = []
    correct_pred = 0
    total_queries = 0

    print(f"Encoding queries for Embeddings classifier...")
    query_texts = [q.get('text', '') for q in queries]
    q_embs = embedder.encode_queries([f"Represent this sentence for searching relevant passages: {t}" for t in query_texts])
    predictions = [classifier.predict(emb) for emb in q_embs]
    prep_time_per_query = 0  # already included in encoding

    print(f"Evaluating Embeddings + classifier (k={k}, threshold={threshold})...")
    for i, q in enumerate(tqdm(queries, desc="Embeddings + Clf")):
        qid = str(q['id'])
        true_cat = q.get('category')
        rel_info = qgts.get(qid, {})
        relevant_ids = {str(item.get("doc_id")) for item in rel_info.get("relevant_doc_ids", [])}
        if not relevant_ids:
            continue

        total_queries += 1
        pred_cat, conf = predictions[i]
        if true_cat and pred_cat == true_cat:
            correct_pred += 1

        start = time.time()
        if conf >= threshold:
            results = embedder.retrieve_category(q_embs[i], pred_cat, top_k=k)
        else:
            results = embedder.retrieve_global(q_embs[i], top_k=k)
        retrieved = [str(doc_id) for doc_id, _ in results]
        latencies.append(time.time() - start)

        hits = [1 if d in relevant_ids else 0 for d in retrieved]
        num_hits = sum(hits)
        all_p.append(num_hits / k)
        all_r.append(num_hits / len(relevant_ids))
        mrr = next((1.0/(idx+1) for idx, h in enumerate(hits) if h), 0.0)
        all_mrr.append(mrr)

    return {
        "Method": "Embeddings (BGE) + Classifier",
        "P@k": np.mean(all_p),
        "R@k": np.mean(all_r),
        "MRR": np.mean(all_mrr),
        "Latency (ms)": np.mean(latencies) * 1000,
        "Classifier Accuracy": correct_pred / total_queries if total_queries else 0.0
    }


## `hybrid_rrf_fusion`

**Purpose**  
Combines two ranked lists (BM25 and dense) using **Reciprocal Rank Fusion (RRF)**.  
Documents that appear in both lists receive a higher priority.

**Parameters**
- `bm25_results`, `embed_results` : lists of `(doc_id, score)` from each retriever.
- `top_k` : number of documents to return after fusion.
- `rrf_k` : constant (usually 60) that controls the rank decay.

**How it works**
1. Converts each result list into a dictionary `{doc_id: rank}`.
2. For every document present in either list:
   - `r_bm` = rank in BM25 (or 1000 if absent)
   - `r_em` = rank in dense (or 1000 if absent)
   - RRF score = `1/(rrf_k + r_bm) + 1/(rrf_k + r_em)`
   - Also stores a boolean `in_both` (true if document appears in both lists).
3. Sorts documents first by `in_both` (true first), then by RRF score descending.
4. Returns the top‑`top_k` document IDs.

In [17]:
def hybrid_rrf_fusion(bm25_results, embed_results, top_k=100, rrf_k=60):
    """Reciprocal Rank Fusion (RRF) with priority to documents present in both lists."""
    scores = {}
    bm_dict = {str(did): rank for rank, (did, _) in enumerate(bm25_results)}
    em_dict = {str(did): rank for rank, (did, _) in enumerate(embed_results)}
    all_ids = set(bm_dict.keys()) | set(em_dict.keys())
    for did in all_ids:
        r_bm = bm_dict.get(did, 1000)
        r_em = em_dict.get(did, 1000)
        score = 1.0/(rrf_k + r_bm) + 1.0/(rrf_k + r_em)
        in_both = (did in bm_dict and did in em_dict)
        scores[did] = (in_both, score)
    sorted_res = sorted(scores.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True)
    return [did for did, _ in sorted_res[:top_k]]


## `evaluate_hybrid_with_classifier`

**Purpose**  
Evaluates the hybrid retrieval system (BM25 + Dense + RRF) with classifier‑guided category filtering.

**Parameters**
- `bm25_retriever`, `embedder`, `classifier`, `queries`, `qgts`, `k`, `threshold` : as before.
- `fetch_k` : number of candidates to retrieve from each retriever before fusion (default 200).

**How it works**
1. Batch‑encodes queries and gets predictions.
2. For each query:
   - If `confidence >= threshold`:
     - Retrieves `fetch_k` documents from BM25 and dense **only within the predicted category**.
   - Else:
     - Retrieves `fetch_k` documents globally from both.
   - Fuses the two lists using `hybrid_rrf_fusion()` to obtain final `k` documents.
3. Computes metrics and classifier accuracy.

**Why `fetch_k` > `k`**  
Fusing more candidates than needed gives the RRF algorithm more material to rank, especially to capture the intersection (documents found by both methods).

In [18]:
def evaluate_hybrid_with_classifier(bm25_retriever, embedder, classifier, queries, qgts,
                                    k=50, threshold=0.80, fetch_k=200):
    """
    Hybrid retrieval (BM25 + Dense + RRF) with category filtering.
    """
    all_p, all_r, all_mrr = [], [], []
    latencies = []
    correct_pred = 0
    total_queries = 0

    print(f"Encoding queries for Hybrid classifier...")
    query_texts = [q.get('text', '') for q in queries]
    q_embs = embedder.encode_queries([f"Represent this sentence for searching relevant passages: {t}" for t in query_texts])
    predictions = [classifier.predict(emb) for emb in q_embs]

    print(f"Evaluating Hybrid + classifier (k={k}, threshold={threshold})...")
    for i, q in enumerate(tqdm(queries, desc="Hybrid + Clf")):
        qid = str(q['id'])
        true_cat = q.get('category')
        rel_info = qgts.get(qid, {})
        relevant_ids = {str(item.get("doc_id")) for item in rel_info.get("relevant_doc_ids", [])}
        if not relevant_ids:
            continue

        total_queries += 1
        pred_cat, conf = predictions[i]
        if true_cat and pred_cat == true_cat:
            correct_pred += 1

        start = time.time()
        if conf >= threshold:
            bm_res = bm25_retriever.retrieve_category(query_texts[i], pred_cat, top_k=fetch_k)
            emb_res = embedder.retrieve_category(q_embs[i], pred_cat, top_k=fetch_k)
        else:
            bm_res = bm25_retriever.retrieve(query_texts[i], top_k=fetch_k)
            emb_res = embedder.retrieve_global(q_embs[i], top_k=fetch_k)

        fused = hybrid_rrf_fusion(bm_res, emb_res, top_k=k)
        latencies.append(time.time() - start)

        hits = [1 if d in relevant_ids else 0 for d in fused]
        num_hits = sum(hits)
        all_p.append(num_hits / k)
        all_r.append(num_hits / len(relevant_ids))
        mrr = next((1.0/(idx+1) for idx, h in enumerate(hits) if h), 0.0)
        all_mrr.append(mrr)

    return {
        "Method": "Hybrid (BM25 + BGE + RRF) + Classifier",
        "P@k": np.mean(all_p),
        "R@k": np.mean(all_r),
        "MRR": np.mean(all_mrr),
        "Latency (ms)": np.mean(latencies) * 1000,
        "Classifier Accuracy": correct_pred / total_queries if total_queries else 0.0
    }

## `compare_all_with_classifier`

**Purpose**  
Orchestrates the evaluation of all retrieval methods (TF‑IDF, BM25+, Embeddings, Hybrid) that use the classifier for category filtering.  
It returns a single DataFrame comparing their performance metrics.

**Parameters**
- `tfidf_vectorizer`, `tfidf_matrix` : TF‑IDF components (if available; can be `None` to skip TF‑IDF).
- `doc_ids` : list of document IDs aligned with the TF‑IDF matrix rows.
- `bm25_retriever` : instance of `BM25Retriever`.
- `embedder` : instance of `EmbeddingRetriever`.
- `classifier` : trained `QueryClassifier`.
- `queries` : list of query dictionaries (with 'id', 'text', 'category').
- `qgts` : ground truth dictionary `{query_id: relevant_doc_ids}`.
- `docs_df` : DataFrame with document categories (for TF‑IDF filtering).
- `k` : number of documents to retrieve per query.
- `threshold` : confidence threshold for expert mode.

**How it works**
1. Initialises an empty list `results`.
2. If TF‑IDF components are provided, calls `evaluate_tfidf_with_classifier()` and appends the result.
3. Calls `evaluate_bm25_with_classifier()` and appends its result.
4. Calls `evaluate_embeddings_with_classifier()` and appends its result.
5. Calls `evaluate_hybrid_with_classifier()` and appends its result.
6. Returns a pandas DataFrame where each row corresponds to one retrieval method and columns contain metrics (P@k, R@k, MRR, latency, classifier accuracy).
 
It provides a single entry point to compare all variants side‑by‑side, making it easy to see which method benefits most from the classifier and to analyse trade‑offs between recall, precision, MRR, and latency.

In [19]:
def compare_all_with_classifier(tfidf_retriever, bm25_retriever, embedder, classifier,
                                queries, qgts, docs_df, k=20, threshold=0.80):
    """
    Run all evaluations (TF‑IDF, BM25+, Embeddings, Hybrid) with classifier.
    """
    results = []

    # TF‑IDF
    if tfidf_retriever is not None:
        res_tfidf = evaluate_tfidf_retriever_with_classifier(
            tfidf_retriever, classifier, embedder, queries, qgts, k=k, threshold=threshold
        )
        results.append(res_tfidf)

    # BM25+
    res_bm = evaluate_bm25_with_classifier(
        bm25_retriever, classifier, embedder, queries, qgts, k=k, threshold=threshold
    )
    results.append(res_bm)

    # Embeddings
    res_emb = evaluate_embeddings_with_classifier(
        embedder, classifier, queries, qgts, k=k, threshold=threshold
    )
    results.append(res_emb)

    # Hybrid
    res_hy = evaluate_hybrid_with_classifier(
        bm25_retriever, embedder, classifier, queries, qgts, k=k, threshold=threshold
    )
    results.append(res_hy)

    return pd.DataFrame(results)

In [20]:
# EXÉCUTION DE LA COMPARAISON AVEC AFFICHAGE PROFESSIONNEL

# data preprocessing
if 'docs_df' not in dir():
    docs_df = pd.DataFrame(corpus)
if 'doc_ids_list' not in dir():
    doc_ids_list = docs_df['id'].astype(str).tolist()
    
# Ensure all retrievers are instantiated 
if 'cat_indexer' not in dir():
    cat_indexer = CategoryIndexer(corpus)

if 'tfidf_retriever' not in dir():
    tfidf_retriever = TFIDFRetriever(corpus, cat_indexer)

if 'bm25_retriever' not in dir():
    bm25_retriever = BM25Retriever(corpus, cat_indexer)

# Run comparison with all four methods
comparison_df = compare_all_with_classifier(
    tfidf_retriever=tfidf_retriever,
    bm25_retriever=bm25_retriever,
    embedder=embedder,
    classifier=classifier,
    queries=q_val_split,
    qgts=relevance,
    docs_df=docs_df,
    k=500,
    threshold=cfg.CAT_THRESHOLD  
)

# Display formatted table
print("\n📊 COMPARISON TABLE: ALL METHODS WITH CLASSIFIER")
display(comparison_df.style.format({
    "P@k": "{:.2%}",
    "R@k": "{:.2%}",
    "MRR": "{:.2%}",
    "Latency (ms)": "{:.2f}ms",
    "Classifier Accuracy": "{:.2%}"
}))

# Tri par MRR décroissant
comparison_df = comparison_df.sort_values(by="MRR", ascending=False).reset_index(drop=True)

def highlight_best(s):
    is_max = s == s.max()
    return ['background-color: lightgreen; font-weight: bold' if v else '' for v in is_max]

styled = comparison_df.style \
    .format({
        "P@k": "{:.2%}",
        "R@k": "{:.2%}",
        "MRR": "{:.2%}",
        "Latency (ms)": "{:.2f} ms",
        "Classifier Accuracy": "{:.2%}"
    }) \
    .apply(highlight_best, subset=["MRR", "R@k", "P@k"]) \
    .background_gradient(subset=["MRR"], cmap="YlGn", low=0.3, high=0.9) \
    .background_gradient(subset=["R@k"], cmap="Blues", low=0.3, high=0.9) \
    .set_properties(**{'text-align': 'center'}) \
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f2f2f2')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
    ]) \
    .set_caption("Performance des modèles de recherche avec classifieur (seuil = 0.80)")

print("\n")
print("TABLEAU COMPARATIF DES MODÈLES (k=20, threshold=0.80)")
display(styled)

# --- Ligne de résumé ---
best_row = comparison_df.iloc[0]
print(f"\nMeilleure méthode : **{best_row['Method']}**")
print(f"   • MRR  = {best_row['MRR']:.2%}")
print(f"   • R@20 = {best_row['R@k']:.2%}")
print(f"   • P@20 = {best_row['P@k']:.2%}")
print(f"   • Latence = {best_row['Latency (ms)']:.1f} ms")
print(f"   • Accuracy du classifieur = {best_row['Classifier Accuracy']:.2%}")

🧠 Encoding queries for TF‑IDF classifier...
🚀 Evaluating TF‑IDF + classifier (k=500, threshold=0.8)...


TF‑IDF+Clf: 100%|██████████| 66/66 [00:03<00:00, 19.13it/s]


Encoding queries for BM25+ classifier...
Evaluating BM25+ + classifier (k=500, threshold=0.8)...


BM25+ + Clf: 100%|██████████| 66/66 [01:53<00:00,  1.72s/it]


Encoding queries for Embeddings classifier...
Evaluating Embeddings + classifier (k=500, threshold=0.8)...


...


📊 COMPARISON TABLE: ALL METHODS WITH CLASSIFIER


,Method,P@k,R@k,MRR,Latency (ms),Classifier Accuracy
0,TF‑IDF + Classifier,0.63%,42.60%,18.61%,51.80ms,90.91%
1,BM25+ + Classifier,0.76%,42.79%,24.80%,1715.24ms,90.91%
2,Embeddings (BGE) + Classifier,1.24%,70.76%,52.02%,30.65ms,90.91%
3,Hybrid (BM25 + BGE + RRF) + Classifier,1.11%,65.85%,42.72%,1665.81ms,90.91%




TABLEAU COMPARATIF DES MODÈLES (k=20, threshold=0.80)


,Method,P@k,R@k,MRR,Latency (ms),Classifier Accuracy
0,Embeddings (BGE) + Classifier,1.24%,70.76%,52.02%,30.65 ms,90.91%
1,Hybrid (BM25 + BGE + RRF) + Classifier,1.11%,65.85%,42.72%,1665.81 ms,90.91%
2,BM25+ + Classifier,0.76%,42.79%,24.80%,1715.24 ms,90.91%
3,TF‑IDF + Classifier,0.63%,42.60%,18.61%,51.80 ms,90.91%



Meilleure méthode : **Embeddings (BGE) + Classifier**
   • MRR  = 52.02%
   • R@20 = 70.76%
   • P@20 = 1.24%
   • Latence = 30.6 ms
   • Accuracy du classifieur = 90.91%


## Interpretation of Results

The comparison table clearly shows that the **"Embeddings (BGE) + Classifier"** method outperforms all others on key metrics: MRR (52.02%), recall (70.76%), and latency (28.8 ms). Here is why.

### 1. Why are embeddings + classifier the best?

- **Semantic quality**: The `BAAI/bge-base-en-v1.5` model (768 dimensions) captures the meaning of queries and documents, even when vocabulary differs. It is far more effective than lexical models (TF‑IDF, BM25) which suffer from exact‑match failures.
- **Efficient classifier**: With 90.91% accuracy on the validation set, the classifier correctly directs the search to the right category for most queries. The 0.80 threshold ensures that only highly confident predictions trigger expert mode, limiting fatal errors.
- **Per‑category FAISS index**: Restricting the search to a single category dramatically reduces noise and improves precision, while being extremely fast (a few milliseconds).

### 2. Why is the hybrid (BM25 + BGE + RRF) less effective?

- **High latency**: Creating a temporary BM25 sub‑index for each query in expert mode is very expensive (over 1.7 seconds on average). This latency makes the method impractical.
- **Suboptimal RRF fusion**: Although RRF combines ranks, it gives too much weight to BM25 when the two lists are of uneven quality. Here BM25 has a much lower recall (42.59% vs 70.76% for embeddings). Fusion therefore dilutes the performance of the embeddings.
- **Fewer gains from the classifier**: Hybrid search in expert mode must create a BM25 sub‑index on the fly, cancelling part of the benefit of category filtering.

### 3. Why are BM25+ and TF‑IDF so weak?

- **Lexical limitations**: These models do not understand synonyms or paraphrases. Their recall is very limited (around 42% even with category filtering).
- **Classifier helps little**: Even when restricting to the correct category, the intrinsic score quality remains low. BM25+’s MRR reaches only 24.55%.
- **High latency for BM25+**: Creating a BM25 sub‑index per query in expert mode is extremely slow (1.7 s).

### 4. Role of the classifier

- **Classifier accuracy: 90.91%** – On the validation set (66 queries), the classifier correctly predicted the category in 60 out of 66 cases. This high score justifies the use of expert mode for confident queries (≥ 0.80).
- **Impact on embeddings**: The MRR gain compared to dense search without a classifier (not measured here, but typically ~50%) is modest but real. The classifier eliminates documents from completely irrelevant categories, improving the precision of the top ranks.
- **Confidence threshold**: Set to 0.80, it avoids fatal errors (expert mode with the wrong category). The 9% of queries in global mode (confidence < 0.80) do not lose recall because they use global search.

### 5. Conclusion

- **Best method**: **Embeddings (BGE) + Classifier** is the optimal choice for this competition. It offers the best recall/MRR/latency trade‑off.
- **Why not use the hybrid?** Excessive latency and the MRR degradation compared to pure embeddings make it unsuitable.
- **Classifier is useful**: Even with an already strong dense model, the classifier brings an improvement (especially in precision and MRR) by restricting the search space to relevant categories.
- **Potential improvements**: We could test other embedding models (e.g., `bge-large`), adjust the confidence threshold, or use a cross‑encoder for final reranking (at the cost of higher latency).

**In summary**, the "dense + classifier" system best exploits the complementarity between the semantics of embeddings and the domain knowledge provided by the classifier.

## Critique of the current solution and areas for improvement

Although our best system (**Embeddings BGE + classifier**) achieves respectable performance (MRR 52%, recall 71%), several limitations remain. We analyze them below, along with possible improvements.

### Identified weaknesses

1. **Under‑used document content for embeddings**  
   - Currently, only the `"text"` field is used to compute embeddings. Titles and tags, which often contain highly discriminative keywords, are ignored.  
   - **Why this choice?** Encoding the full corpus (216,000 documents) with the BGE model already takes about 1 hour on a GPU. Adding title and tags would have significantly increased this time (longer text → more tokens to process).  
   - **Consequence**: we chose to keep this version due to time constraints, but we are well aware that the scores would have been better (estimated gain of 2‑5 MRR points) if we had included `title` and `tags`.

2. **Single embedding model**  
   - `bge-base-en-v1.5` (768 dims) is strong, but more recent or specialised models exist (e.g., `bge-large-en-v1.5`, `e5-mistral-7b`, `Snowflake-arctic-embed-m`).  
   - *Impact*: a better model could improve recall and MRR, at the cost of increased computation time.

3. **Classifier trained only on queries**  
   - Validation accuracy of 90.9% – decent but not perfect. Errors (especially on minority classes) can cause failures in expert mode.  
   - *Impact*: a few misclassified queries with high confidence → loss of relevant documents.

4. **Fixed confidence threshold (0.80)**  
   - A single threshold does not account for query‑specific difficulty or probability distribution.  
   - *Impact*: easy queries could benefit from a lower threshold, while ambiguous queries might need a higher one.

5. **No fine‑grained reranking (cross‑encoder)**  
   - The system relies only on cosine similarity of embeddings. A cross‑encoder (e.g., `ms-marco-MiniLM-L-6-v2`) could re‑score the top‑100 candidates with higher precision.  
   - *Impact*: MRR could be increased by several points (at the cost of higher latency).

6. **Inefficient hybrid (BM25 + BGE)**  
   - Creating a temporary BM25 sub‑index for each query in expert mode is extremely slow (>1.7 s) and degrades MRR compared to pure embeddings.  
   - *Idea*: pre‑compute per‑category BM25 sub‑indices to make hybrid viable.

### Concrete improvement directions

| Area | Action | Expected gain | Cost |
|------|--------|---------------|------|
| **Richer content** | Include `title` (duplicated) and `tags` in document text for embeddings. | +2‑5% MRR | Re‑encoding (1‑2 h) |
| **Better embedding model** | Test `bge-large-en-v1.5` or `Snowflake-arctic-embed-l`. | +3‑8% MRR | Higher memory/time |
| **More robust classifier** | Use a small LLM (e.g., `gemini‑flash`) for zero‑shot or fine‑tune a stronger model. | +5% accuracy | API or training |
| **Adaptive threshold** | Adjust threshold per class or use probability as a continuous weight. | +1‑2% MRR | Low |
| **Cross‑encoder reranking** | Add a cross‑encoder on the top‑100 dense results. | +5‑10% MRR | 10× latency |
| **Ensemble of dense models** | Fuse scores from `bge-base` and `all-MiniLM-L12-v2` using RRF or weighted average. | +2‑4% MRR | Two FAISS indices |

### Concluding critique

Our current solution is a reasonable trade‑off between performance and computation time, but it is not optimal. **The main limitation** is the absence of titles and tags in the embeddings – we would very likely have achieved a better public score if we had taken the time to include them. In the future, we would plan the corpus encoding with enriched content and explore cross‑encoder reranking to push MRR beyond 55%.

## Submission Generation – Embeddings + Classifier

**Purpose**  
Creates the final Kaggle submission CSV using the best performing method: dense retrieval (BGE embeddings + FAISS) guided by the domain classifier.

**How it works**
1. Encodes all test queries in batch using the BGE model (with the instruction prefix).
2. Classifies each query (category + confidence) using the trained classifier.
3. For each query:
   - If confidence ≥ threshold → **expert mode**: searches only within the predicted category (using the pre‑built FAISS sub‑index).
   - Else → **safety mode**: searches the entire corpus (global FAISS index).
4. Retrieves top‑k document IDs.
5. Exports a CSV with columns: `query_id`, `relevant_doc_ids` (JSON list), `category`.

**Parameters**
- `embedder` : `EmbeddingRetriever` (BGE + FAISS)
- `classifier` : trained `QueryClassifier`
- `test_queries` : list of test query dictionaries
- `cfg` : configuration object with `OUTPUT_DIR`
- `k` : number of documents to retrieve per query (e.g., 500)
- `threshold` : confidence threshold (optimised on validation set, e.g., 0.80)

**Output**  
CSV file named `submission_embeddings_classifier_t{threshold}_k{k}.csv` in `cfg.OUTPUT_DIR`.

In [21]:
def generate_submission_embeddings_classifier(embedder, classifier, test_queries, cfg, k=500, threshold=0.80):
    """
    Generates submission CSV using the best method: dense retrieval (FAISS) 
    with classifier-guided category filtering (when confidence ≥ threshold).
    
    Parameters:
    - embedder: EmbeddingRetriever instance (BGE + FAISS)
    - classifier: trained QueryClassifier
    - test_queries: list of test query dictionaries (with 'id', 'text')
    - cfg: Config object (contains OUTPUT_DIR)
    - k: number of documents to retrieve per query
    - threshold: confidence threshold for expert mode
    
    Returns:
    - Path to the generated CSV file
    """
    output_dir = Path(cfg.OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"submission_embeddings_classifier_t{threshold}_k{k}.csv"
    
    rows = []
    
    # Batch encoding and classification
    query_texts = [q.get('text', '') for q in test_queries]
    print(f"Encoding and classifying {len(test_queries)} test queries...")
    q_embs = embedder.encode_queries(query_texts)
    predictions = [classifier.predict(emb) for emb in q_embs]
    
    print(f"Generating results (k={k}, threshold={threshold})...")
    
    for i, query_obj in enumerate(tqdm(test_queries, desc="Processing")):
        query_id = str(query_obj['id'])
        q_emb = q_embs[i]
        pred_cat, confidence = predictions[i]
        
        # Index selection based on confidence
        if confidence >= threshold:
            # Expert mode: search only within predicted category
            results = embedder.retrieve_category(q_emb, pred_cat, top_k=k)
        else:
            # Safety mode: global search
            results = embedder.retrieve_global(q_emb, top_k=k)
        
        final_ids = [str(rid) for rid, _ in results]
        
        rows.append({
            "query_id": query_id,
            "relevant_doc_ids": json.dumps(final_ids),
            "category": str(pred_cat)
        })
    
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    
    print(f"\nSubmission saved: {out_path}")
    return out_path


# Generate submission using the best method: Embeddings + Classifier
final_submission_path = generate_submission_embeddings_classifier(
    embedder=embedder,
    classifier=classifier,
    test_queries=test_queries,
    cfg=cfg,
    k=500,          
    threshold=cfg.CAT_THRESHOLD      
)

print(f"\nSubmission file ready: {final_submission_path}")

Encoding and classifying 141 test queries...
Generating results (k=500, threshold=0.8)...


Processing: 100%|██████████| 141/141 [00:03<00:00, 36.86it/s]


Submission saved: /kaggle/working/submission_embeddings_classifier_t0.8_k500.csv

Submission file ready: /kaggle/working/submission_embeddings_classifier_t0.8_k500.csv
